In [ ]:
import cv2
import numpy as np
import pandas as pd
import tensorflow as tf
import os
import joblib
import os
import django
import sys

sys.path.append('../backend')

os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'backend.settings')
django.setup()

from accounts.ml_model import model, DEPLOY_IMG_SIZE, CLASS_NAMES

from tensorflow.keras.applications.efficientnet import preprocess_input

# ==========================================
# 1. LOAD PRE-TRAINED SCALERS (RELATIVE PATHS)
# ==========================================
# This ensures it works on your local machine and your teammate's machine
BASE_DIR = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
MODEL_DIR = os.path.join(BASE_DIR, 'model')

try:
    age_scaler = joblib.load(os.path.join(MODEL_DIR, 'age_scaler.pkl'))
    sex_encoder = joblib.load(os.path.join(MODEL_DIR, 'sex_encoder.pkl'))
except FileNotFoundError:
    print("Warning: Scaler files not found in /model/ directory.")

# ==========================================
# 2. IMAGE PREPROCESSING PIPELINE
# ==========================================
def apply_clahe(img):
    """Enhances micro-lesions using LAB-space CLAHE."""
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=1.2, tileGridSize=(16, 16))
    cl = clahe.apply(l)
    return cv2.cvtColor(cv2.merge((cl, a, b)), cv2.COLOR_LAB2RGB)

def prepare_inputs(image_bytes, age, sex):
    # --- Image Branch ---
    img_np = np.frombuffer(image_bytes, np.uint8)
    img = cv2.imdecode(img_np, cv2.IMREAD_COLOR)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # Resize and CLAHE
    img_res = cv2.resize(img_rgb, (DEPLOY_IMG_SIZE, DEPLOY_IMG_SIZE))
    img_final = apply_clahe(img_res)
    
    img_preprocessed = preprocess_input(np.expand_dims(img_final, axis=0).astype('float32'))

    # --- Clinical Branch ---
    # Convert 'M'/'F' to 'Male'/'Female' to match your scaler's training data
    sex_map = {"M": "Male", "F": "Female"}
    clinical_df = pd.DataFrame([[float(age), sex_map.get(sex, sex)]], 
                               columns=['Patient Age', 'Patient Sex'])
    
    age_scaled = age_scaler.transform(clinical_df[['Patient Age']])[0][0]
    sex_encoded = sex_encoder.transform(clinical_df[['Patient Sex']]) # Returns [0,1] or [1,0]
    
    # Combined into shape (1, 3) -> [Age, Sex_F, Sex_M]
    clinical_input = np.array([[age_scaled, sex_encoded[0][0], sex_encoded[0][1]]], dtype="float32")

    return img_final, img_preprocessed, clinical_input

# ==========================================
# 3. GRAD-CAM & CLINICAL LOGIC
# ==========================================
def get_prediction_and_heatmap(img_pre, cli_pre):
    """Applies Clinical Thresholds and calculates Grad-CAM Heatmap."""
    
    # Locate the final conv layer of EfficientNetB3
    grad_model = tf.keras.models.Model(
        inputs=model.inputs,
        outputs=[model.get_layer("top_conv").output, model.output]
    )

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model([img_pre, cli_pre])
        
        # Clinical Threshold Logic (Prioritize Safety)
        probs = predictions[0].numpy()
        if probs[4] >= 0.25:   idx = 4  # Proliferative
        elif probs[3] >= 0.35: idx = 3  # Severe
        else:                  idx = np.argmax(probs)
        
        target_output = predictions[:, idx]

    # Compute Gradients
    grads = tape.gradient(target_output, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    
    # Generate Heatmap
    heatmap = tf.reduce_sum(conv_outputs[0] * pooled_grads, axis=-1)
    heatmap = np.maximum(heatmap, 0)
    if np.max(heatmap) > 0:
        heatmap /= np.max(heatmap)
    
    return heatmap, idx, float(probs[idx])

# ==========================================
# 4. MAIN EXECUTION ENTRY
# ==========================================
def run_prediction(image_bytes, age, sex):
    img_disp, img_pre, cli_pre = prepare_inputs(image_bytes, age, sex)
    heatmap, pred_idx, confidence = get_prediction_and_heatmap(img_pre, cli_pre)

    # Apply Heatmap Overlay
    heatmap_res = cv2.resize(heatmap, (img_disp.shape[1], img_disp.shape[0]))
    heatmap_color = cv2.applyColorMap(np.uint8(255 * heatmap_res), cv2.COLORMAP_JET)
    overlay_img = cv2.addWeighted(img_disp, 0.6, cv2.cvtColor(heatmap_color, cv2.COLOR_BGR2RGB), 0.4, 0)

    return {
        "stage": CLASS_NAMES[pred_idx],
        "confidence": round(confidence * 100, 2),
        "visual_result": overlay_img,
        "is_high_risk": pred_idx >= 3
    }